In [50]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.font_manager import FontProperties
import os
from tqdm import tqdm

# ==== 中文字体配置 ====
font_path = r"C:\Windows\Fonts\msyh.ttc"
font_prop = FontProperties(fname=font_path)
plt.rcParams['font.family'] = font_prop.get_name()
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['savefig.facecolor'] = 'white'

# ==== 数据加载 ====
columns = [
    'user_id', 'item_id', 'category_id', 'behavior_type', 'timestamp'
]

# 自适应加载数据（含表头检测）
try:
    df = pd.read_csv(
        r'D:\data_analysis\UserBehavior.csv',
        names=columns,
        header=0,  # 优先使用文件自带表头
        nrows=100000
    )
except pd.errors.ParserError:
    df = pd.read_csv(
        r'D:\data_analysis\UserBehavior.csv',
        names=columns,
        header=None,
        nrows=100000
    )

# 验证列名
print("修正后列名：", df.columns.tolist())
print("\n样本数据预览：")
print(df.head(2))

# ==== 时间戳处理 ====
# 自适应时间戳单位
def parse_timestamp(ts):
    try:
        return pd.to_datetime(ts, unit='s')
    except OverflowError:
        return pd.to_datetime(ts, unit='ms')

df['datetime'] = df['timestamp'].apply(parse_timestamp)
df['date'] = df['datetime'].dt.date
df['hour'] = df['datetime'].dt.hour

# 验证时间处理
print("\n时间处理验证：")
print(df[['timestamp', 'datetime', 'hour']].head())

# ==== 用户行为分析 ====
behavior_counts = df['behavior_type'].value_counts()
print("\n行为类型分布：")
print(behavior_counts)

hourly_activity = df['hour'].value_counts().sort_index()

# 绘制用户活跃时段分布图
plt.figure(figsize=(12,6))
hourly_activity.plot(kind='bar', color='steelblue')
plt.title('用户活跃时段分布（按小时）', fontproperties=font_prop, fontsize=14)
plt.xlabel('小时', fontproperties=font_prop, fontsize=12)
plt.ylabel('行为次数', fontproperties=font_prop, fontsize=12)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(r'D:\data_analysis', 'hourly_activity.png'), dpi=300)
plt.close()

# ==== 用户行为转化漏斗 ====
behavior_order = ['pv', 'fav', 'cart', 'buy']
funnel_data = df.groupby('behavior_type')['user_id'].nunique().reindex(behavior_order)
conversion_rate = (funnel_data / funnel_data['pv']).round(3)

# 绘制转化漏斗图
plt.figure(figsize=(12,6))
plt.plot(funnel_data.values, marker='o', linestyle='--', color='#d62728')
plt.title('用户行为转化漏斗分析', fontproperties=font_prop, fontsize=14)
plt.xticks(range(4), ['浏览', '收藏', '加购', '购买'], fontproperties=font_prop, fontsize=12)
plt.ylabel('独立用户数', fontproperties=font_prop, fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()

# 添加注释（仅保存一次）
for i, rate in enumerate(conversion_rate):
    plt.text(i, funnel_data.iloc[i], f'{rate*100:.1f}%', 
             ha='center', va='bottom', fontsize=12, color='green')

plt.savefig(os.path.join(r'D:\data_analysis', 'conversion_funnel.png'), dpi=300)
plt.close()

# ==== RFM分析 ====
purchase_data = df[df['behavior_type'] == 'buy']
current_date = purchase_data['datetime'].max()
rfm = purchase_data.groupby('user_id').agg(
    R=('datetime', lambda x: (current_date - x.max()).days),
    F=('item_id', 'count'),
    M=('category_id', 'nunique')
)

def safe_qcut(series, q, labels, default_labels):
    try:
        return pd.qcut(series, q=q, labels=labels, duplicates='drop')
    except ValueError:
        return pd.cut(series, bins=q, labels=default_labels)

rfm['R_score'] = safe_qcut(rfm['R'], q=5, labels=range(5,0,-1), default_labels=range(5,0,-1))
rfm['F_score'] = safe_qcut(rfm['F'], q=5, labels=range(1,6), default_labels=range(1,6))
rfm['M_score'] = safe_qcut(rfm['M'], q=5, labels=range(1,6), default_labels=range(1,6))
rfm['RFM'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str) + rfm['M_score'].astype(str)
rfm.to_csv(r'D:\data_analysis\user_rfm.csv', index=False)

# 绘制R值分布
plt.figure(figsize=(10,6))
rfm['R'].hist(bins=20)
plt.title('最近购买间隔分布', fontproperties=font_prop)
plt.xlabel('天数', fontproperties=font_prop)
plt.ylabel('用户数', fontproperties=font_prop)
plt.savefig(os.path.join(r'D:\data_analysis', 'r_dist.png'))
plt.close()

# ==== 成果展示 ====
save_dir = r'D:\data_analysis'
os.makedirs(save_dir, exist_ok=True)

# 生成 HTML 报告
report_html = f"""
<h2>用户行为分析报告</h2>
<h3>核心指标</h3>
<ul>
  <li>高峰时段：{hourly_activity.idxmax()}时</li>
  <li>最终转化率：{conversion_rate['buy']*100:.1f}%</li>
</ul>

<h3>可视化图表</h3>
<figure>
  <img src="./hourly_activity.png" alt="用户活跃时段分布" width="600">
  <figcaption>用户活跃时段分布（按小时）</figcaption>
</figure>

<figure>
  <img src="./conversion_funnel.png" alt="用户行为转化漏斗" width="600">
  <figcaption>用户行为转化漏斗分析</figcaption>
</figure>
"""

report_html += """
<h3>业务优化建议</h3>
<ol>
  <li><strong>高峰时段运营</strong>：在{hourly_activity.idxmax()}时增加促销活动，提升用户粘性。</li>
  <li><strong>转化率优化</strong>：针对收藏→加购流失用户，推送个性化优惠券（如满减/赠品）。</li>
  <li><strong>高价值用户维护</strong>：对RFM评分高的用户提供专属客服和优先体验新功能。</li>
</ol>
"""

with open(os.path.join(save_dir, 'report.html'), 'w', encoding='utf-8') as f:
    f.write(report_html)

修正后列名： ['user_id', 'item_id', 'category_id', 'behavior_type', 'timestamp']

样本数据预览：
   user_id  item_id  category_id behavior_type   timestamp
0        1  2333346      2520771            pv  1511561733
1        1  2576651       149192            pv  1511572885

时间处理验证：
    timestamp            datetime  hour
0  1511561733 2017-11-24 22:15:33    22
1  1511572885 2017-11-25 01:21:25     1
2  1511593493 2017-11-25 07:04:53     7
3  1511596146 2017-11-25 07:49:06     7
4  1511616481 2017-11-25 13:28:01    13

行为类型分布：
behavior_type
pv      89709
cart     5446
fav      2744
buy      2101
Name: count, dtype: int64
